# CrediSense AI — Dataset Diagnostics
## Notebook 01: Empirical Analysis of Credit Risk Dataset

This notebook reproduces all figures and statistics referenced in `DATASET_ANALYSIS.md`.  
Run cells in order. The model and dataset must be present at `models/model.pkl` and `data/loan_cleaned.csv`.

**Contents:**
1. Setup & Data Loading
2. Class Imbalance Analysis
3. Feature Distribution Diagnostics
4. Model Evaluation (ROC, PR, Calibration, Brier)
5. Error Decomposition (FP/FN cost analysis)
6. SHAP Global Importance
7. Fairness Audit (Disparate Impact Ratio)

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

BASE_DIR = Path().resolve().parent  # adjust if running from notebooks/
sys.path.insert(0, str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

from src.data_loader import load_data
from src.preprocessing import clean_data
from src.feature_engineering import create_features
from sklearn.model_selection import train_test_split

print('Imports OK')

## 1. Setup & Data Loading

In [ ]:
model = joblib.load(BASE_DIR / 'models' / 'model.pkl')

df_raw = load_data(str(BASE_DIR / 'data' / 'loan_cleaned.csv'))
df     = clean_data(df_raw)
df     = create_features(df)

X = df.drop(columns=['Risk_Flag'])
y = df['Risk_Flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Full dataset : {df.shape}')
print(f'Training set : {X_train.shape}')
print(f'Test set     : {X_test.shape}')
print(f'Default rate : {y.mean():.2%}')

## 2. Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Absolute counts
counts = y.value_counts()
axes[0].bar(['Non-Default (0)', 'Default (1)'], counts.values,
            color=['#2ca02c', '#d62728'], alpha=0.85, edgecolor='white')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontsize=11)
axes[0].set_title('Class Distribution — Absolute Counts')
axes[0].set_ylabel('Number of Records')

# Imbalance ratio over profession
prof_dr = df.groupby('Profession')['Risk_Flag'].mean().sort_values(ascending=False).head(15)
axes[1].barh(prof_dr.index, prof_dr.values, color='#d62728', alpha=0.8)
axes[1].axvline(y.mean(), color='grey', linestyle='--', lw=1.5,
                label=f'Overall default rate ({y.mean():.1%})')
axes[1].set_xlabel('Default Rate')
axes[1].set_title('Default Rate by Profession (Top 15)')
axes[1].legend()

plt.tight_layout()
plt.savefig(BASE_DIR / 'assets' / 'class_imbalance.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Class ratio (non-default:default): {counts[0]/counts[1]:.1f}:1')
print(f'Minority class share: {counts[1]/len(y):.2%}')

## 3. Feature Distribution Diagnostics

In [ ]:
numeric_features = ['Income', 'Age', 'Experience', 'CURRENT_JOB_YRS', 'CURRENT_HOUSE_YRS',
                    'income_per_job_year', 'experience_ratio', 'stability_score']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    for label, color in [(0, '#2ca02c'), (1, '#d62728')]:
        axes[i].hist(df[df['Risk_Flag'] == label][feat].dropna(),
                     bins=30, alpha=0.55, color=color,
                     label=f'Risk={label}', density=True)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=10)
    axes[i].set_xlabel('Value')
    if i == 0:
        axes[i].legend(fontsize=8)

fig.suptitle('Feature Distributions by Default Status', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(BASE_DIR / 'assets' / 'feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Model Evaluation — ROC, PR, Calibration, Brier

In [ ]:
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, brier_score_loss, f1_score
)
from sklearn.calibration import calibration_curve

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

fpr, tpr, _  = roc_curve(y_test, y_prob)
auc          = roc_auc_score(y_test, y_prob)
prec, rec, _ = precision_recall_curve(y_test, y_prob)
pr_auc       = average_precision_score(y_test, y_prob)
brier        = brier_score_loss(y_test, y_prob)
gini         = 2 * auc - 1
ks           = float(np.max(tpr - fpr))
f1           = f1_score(y_test, y_pred)

print(f'ROC-AUC : {auc:.4f}')
print(f'Gini    : {gini:.4f}')
print(f'KS      : {ks:.4f}')
print(f'PR-AUC  : {pr_auc:.4f}')
print(f'F1      : {f1:.4f}')
print(f'Brier   : {brier:.4f}')

# Brier Skill Score
bs_clim = float(y_test.mean()) * (1 - float(y_test.mean()))
bss     = 1 - brier / bs_clim
print(f'BSS     : {bss:.4f}  (negative = uncalibrated probabilities)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC
axes[0].plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'LightGBM  AUC={auc:.4f}')
axes[0].plot([0,1],[0,1], color='grey', lw=1.2, linestyle='--', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#1f77b4')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# PR
baseline = float(y_test.mean())
axes[1].step(rec, prec, color='#ff7f0e', lw=2.5, where='post',
             label=f'LightGBM  PR-AUC={pr_auc:.4f}')
axes[1].axhline(baseline, color='grey', lw=1.2, linestyle='--',
                label=f'Baseline = {baseline:.2%}')
axes[1].fill_between(rec, prec, step='post', alpha=0.08, color='#ff7f0e')
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

# Calibration
frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)
axes[2].plot(mean_pred, frac_pos, 's-', color='#2ca02c', lw=2, label='LightGBM')
axes[2].plot([0,1],[0,1], color='grey', lw=1.2, linestyle='--', label='Perfect calibration')
axes[2].set_title(f'Calibration Curve  (Brier={brier:.4f})')
axes[2].set_xlabel('Mean Predicted Probability')
axes[2].set_ylabel('Fraction of Positives')
axes[2].legend()

fig.suptitle('Model Evaluation — 20% Holdout Set (n=50,400)', fontsize=13)
plt.tight_layout()
plt.savefig(BASE_DIR / 'assets' / 'model_evaluation.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Error Decomposition — FP/FN Cost Analysis

In [ ]:
from sklearn.metrics import confusion_matrix

LOAN_AMOUNT  = 500_000  # Rs
LGD          = 0.60     # Loss Given Default
ANNUAL_RATE  = 0.12     # Annual interest rate

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fn_cost = fn * LOAN_AMOUNT * LGD
fp_cost = fp * LOAN_AMOUNT * ANNUAL_RATE  # annual foregone revenue

print(f'Confusion matrix at threshold 0.5:')
print(f'  TN={tn:,}  FP={fp:,}')
print(f'  FN={fn:,}  TP={tp:,}')
print()
print(f'False Negatives (missed defaults): {fn:,}')
print(f'  → Estimated loss: Rs {fn_cost/1e7:.2f} Cr')
print()
print(f'False Positives (wrongful rejections): {fp:,}')
print(f'  → Foregone revenue (annual): Rs {fp_cost/1e7:.2f} Cr')

thresholds  = np.arange(0.1, 0.9, 0.05)
fn_costs    = []
fp_costs    = []
net_costs   = []
for t in thresholds:
    yp_t = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, yp_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    fn_costs.append(fn_t * LOAN_AMOUNT * LGD / 1e7)
    fp_costs.append(fp_t * LOAN_AMOUNT * ANNUAL_RATE / 1e7)
    net_costs.append((fn_t * LOAN_AMOUNT * LGD + fp_t * LOAN_AMOUNT * ANNUAL_RATE) / 1e7)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, fn_costs, 'o-', color='#d62728', lw=2, label='FN Cost — missed defaults (Rs Cr)')
ax.plot(thresholds, fp_costs, 's-', color='#1f77b4', lw=2, label='FP Cost — foregone revenue (Rs Cr)')
ax.plot(thresholds, net_costs, '^-', color='#9467bd', lw=2, linestyle='--', label='Total Cost (Rs Cr)')
optimal = thresholds[np.argmin(net_costs)]
ax.axvline(optimal, color='grey', linestyle=':', lw=1.5, label=f'Min total cost at t={optimal:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Cost (Rs Crore)')
ax.set_title('Threshold vs. Business Cost (FN + FP)')
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'assets' / 'threshold_cost_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'\nOptimal threshold (min total cost): {optimal:.2f}')

## 6. SHAP Global Feature Importance

In [ ]:
import shap

pre  = model.named_steps['preprocessor']
mod  = model.named_steps['model']
feat_names = pre.get_feature_names_out().tolist()

sample = X_test.head(300)
X_t    = pre.transform(sample)
if hasattr(X_t, 'toarray'):
    X_t = X_t.toarray()

explainer = shap.TreeExplainer(mod)
sv        = explainer.shap_values(X_t)
sv        = sv[1] if isinstance(sv, list) else sv

clean_names = [n.replace('num__','').replace('cat__','').replace('_',' ').title()
               for n in feat_names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Beeswarm
plt.sca(axes[0])
shap.summary_plot(sv, X_t, feature_names=clean_names, show=False,
                  plot_size=None, max_display=15)
axes[0].set_title('SHAP Beeswarm — Global Feature Importance', fontsize=12)

# Bar chart (mean |SHAP|)
mean_shap = np.abs(sv).mean(axis=0)
top_idx   = np.argsort(mean_shap)[::-1][:15]
axes[1].barh([clean_names[i] for i in top_idx[::-1]],
             mean_shap[top_idx[::-1]],
             color='#1f77b4', alpha=0.85)
axes[1].set_xlabel('Mean |SHAP Value|')
axes[1].set_title('Top 15 Features by Mean Impact', fontsize=12)

plt.tight_layout()
plt.savefig(BASE_DIR / 'assets' / 'shap_beeswarm.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Fairness Audit — Disparate Impact Ratio

In [ ]:
from src.fairness import subgroup_metrics, disparate_impact_ratio

df_test = X_test.copy()
df_test['Risk_Flag'] = y_test.values

metrics_df = subgroup_metrics(model, df_test)
dir_df     = disparate_impact_ratio(metrics_df)

print('Subgroup Performance:')
print(metrics_df.to_string(index=False))
print()
print('Disparate Impact Ratios:')
print(dir_df.to_string(index=False))

In [ ]:
if not metrics_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Approval rate by subgroup
    palette = ['#1f77b4','#2ca02c','#ff7f0e','#9467bd']
    groups  = metrics_df['group_col'].unique()
    for i, grp in enumerate(groups):
        sub = metrics_df[metrics_df['group_col'] == grp].sort_values('group_val')
        axes[0].bar(sub['group_val'], sub['approval_rate'],
                    label=grp, alpha=0.82, color=palette[i % len(palette)])
    axes[0].axhline(0.80, color='#d62728', linestyle='--', lw=1.8,
                    label='4/5ths threshold (0.80)')
    axes[0].set_xlabel('Subgroup')
    axes[0].set_ylabel('Approval Rate')
    axes[0].set_title('Approval Rate by Subgroup')
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0, 1.05)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

    # DIR bar chart
    if not dir_df.empty:
        colors = ['#2ca02c' if v == 'Yes' else '#d62728'
                  for v in dir_df['fair']]
        axes[1].bar(dir_df['group_col'], dir_df['disparate_impact_ratio'],
                    color=colors, alpha=0.85)
        axes[1].axhline(0.80, color='grey', linestyle='--', lw=1.8,
                        label='Compliance threshold (0.80)')
        axes[1].set_xlabel('Attribute')
        axes[1].set_ylabel('Disparate Impact Ratio')
        axes[1].set_title('DIR per Attribute — Green = Compliant')
        axes[1].set_ylim(0, 1.2)
        axes[1].legend()
        for j, row in dir_df.iterrows():
            axes[1].text(j, row['disparate_impact_ratio'] + 0.02,
                         f"{row['disparate_impact_ratio']:.3f}", ha='center', fontsize=10)

    fig.suptitle('Fairness Audit — ECOA 4/5ths Rule Compliance', fontsize=13)
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'assets' / 'fairness_audit.png', bbox_inches='tight', dpi=150)
    plt.show()

## Summary

| Metric | Value |
|--------|-------|
| ROC-AUC | 0.8049 |
| Gini | 0.6098 |
| KS | 0.4906 |
| PR-AUC | 0.4079 |
| Brier Score | 0.2038 |
| DIR (all groups) | ≥ 0.80 ✅ |

All assets saved to `assets/`. See `DATASET_ANALYSIS.md` for full interpretation.